map target labels, scale the data, and built the 10-beat sliding windows  

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [10]:
ruta_excel = '../data/2_extracted/features_extracted.csv'

# load file
features_extracted = pd.read_csv(ruta_excel)
features_extracted

,record_id,sample,raw_symbol,label,RR_current,RR_prev,RR_ratio,RR_local_mean,R_amplitude,QRS_duration,QRS_energy,ST_mean
0,100,77,N,N,798.888889,798.888889,1.000000,798.888889,0.840,6.0,6.794950,-0.342469
1,100,370,N,N,813.888889,798.888889,1.009301,806.388889,0.940,8.0,8.952100,-0.409691
2,100,662,N,N,811.111111,813.888889,0.998291,812.500000,0.885,6.0,9.148300,-0.393580
3,100,946,N,N,788.888889,811.111111,0.980437,804.629630,0.810,6.0,9.521900,-0.364815
4,100,1231,N,N,791.666667,788.888889,0.987868,801.388889,0.820,6.0,8.076100,-0.403025
...,...,...,...,...,...,...,...,...,...,...,...,...
30157,219,648690,N,N,813.888889,747.222222,0.936102,869.444444,1.765,13.0,29.406025,-0.486173
30158,219,648836,V,V,405.555556,813.888889,0.610879,663.888889,1.580,13.0,33.546200,-0.761358
30159,219,649174,N,N,938.888889,405.555556,1.292543,726.388889,1.275,11.0,33.936025,-1.441481
30160,219,649448,N,N,761.111111,938.888889,1.042816,729.861111,1.450,11.0,29.501425,-0.948272


In [14]:
# =====================================================================
# 1. LABEL MAPPING AND REMOVAL OF UNMAPPED SYMBOLS (AAMI)
# =====================================================================

# Definition from the official dictionary in accordance with the specified AAMI standard
label_map = {
 # N — Normal
 'N': 'N', '.': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
 # S — Supraventricular ectopic
 'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
 # V — Ventricular ectopic
 'V': 'V', 'E': 'V',
 # All other symbols: discard
}

# Identify and map the labels
features_extracted['label'] = features_extracted['raw_symbol'].map(label_map)

# Record and report discarded heartbeats (those that remain as NaN)
discarded_df = features_extracted[features_extracted['label'].isna()]
total_discarded = len(discarded_df)
discarded_counts = discarded_df['raw_symbol'].value_counts()

print("==================================================")
print("FILTRATION AND MAPPING REPORT (AAMI STANDARD)")
print("==================================================")
print(f"Total heartbeats initially: {len(features_extracted)}")
print(f"Total heartbeats excluded: {total_discarded}")
print("\nBreakdown of discarded symbols:")
print(discarded_counts if total_discarded > 0 else "None")

FILTRATION AND MAPPING REPORT (AAMI STANDARD)
Total heartbeats initially: 30162
Total heartbeats excluded: 0

Breakdown of discarded symbols:
None


In [15]:
# Create a clean DataFrame by removing rows without a valid mapping
df_cleaned = features_extracted.dropna(subset=['label']).copy()

df_cleaned

,record_id,sample,raw_symbol,label,RR_current,RR_prev,RR_ratio,RR_local_mean,R_amplitude,QRS_duration,QRS_energy,ST_mean
0,100,77,N,N,798.888889,798.888889,1.000000,798.888889,0.840,6.0,6.794950,-0.342469
1,100,370,N,N,813.888889,798.888889,1.009301,806.388889,0.940,8.0,8.952100,-0.409691
2,100,662,N,N,811.111111,813.888889,0.998291,812.500000,0.885,6.0,9.148300,-0.393580
3,100,946,N,N,788.888889,811.111111,0.980437,804.629630,0.810,6.0,9.521900,-0.364815
4,100,1231,N,N,791.666667,788.888889,0.987868,801.388889,0.820,6.0,8.076100,-0.403025
...,...,...,...,...,...,...,...,...,...,...,...,...
30157,219,648690,N,N,813.888889,747.222222,0.936102,869.444444,1.765,13.0,29.406025,-0.486173
30158,219,648836,V,V,405.555556,813.888889,0.610879,663.888889,1.580,13.0,33.546200,-0.761358
30159,219,649174,N,N,938.888889,405.555556,1.292543,726.388889,1.275,11.0,33.936025,-1.441481
30160,219,649448,N,N,761.111111,938.888889,1.042816,729.861111,1.450,11.0,29.501425,-0.948272


In [16]:
# =====================================================================
# 2. STRATIFIED SPLIT AND SCALING (SINGLE BEAT)
# =====================================================================
feature_cols = ['RR_current', 'RR_prev', 'RR_ratio', 'RR_local_mean', 
                'R_amplitude', 'QRS_duration', 'QRS_energy', 'ST_mean']

train_idx, test_idx = train_test_split(
    df_cleaned.index, 
    test_size=0.20, 
    stratify=df_cleaned['label'], 
    random_state=42
)

# Adjust the scale ONLY with training
scaler = StandardScaler()
scaler.fit(df_cleaned.loc[train_idx, feature_cols])

# Transform the entire dataset and add it to the data frame to ensure traceability
scaled_features = scaler.transform(df_cleaned[feature_cols])
scaled_cols = [f"{col}_scaled" for col in feature_cols]
df_cleaned[scaled_cols] = scaled_features

# Single Beat Sets  
X_train_s = df_cleaned.loc[train_idx, scaled_cols].values
y_train_s = df_cleaned.loc[train_idx, 'label'].values
X_test_s = df_cleaned.loc[test_idx, scaled_cols].values
y_test_s = df_cleaned.loc[test_idx, 'label'].values

# =====================================================================
# 3. CONSTRUCTION OF SLIDING WINDOWS PRESERVING THE ORIGINAL SPLIT
# =====================================================================
X_train_w, y_train_w = [], []
X_test_w, y_test_w = [], []

# We group data by patient to avoid mixing data from different patients
for record_id, group in df_cleaned.groupby('record_id'):
    group_features = group[scaled_cols].values
    group_labels = group['label'].values
    group_indices = group.index.values # Original indices of the DataFrame
    
    if len(group_features) >= 10:
        for i in range(9, len(group_features)):
            window = group_features[i-9 : i+1] # Window T=10
            label = group_labels[i]            # final beat class
            actual_global_idx = group_indices[i] # Overall rating of this heartbeat
            
            # We decide whether it goes to Train or Test based on the original split of beat 'i'
            if actual_global_idx in train_idx:
                X_train_w.append(window)
                y_train_w.append(label)
            elif actual_global_idx in test_idx:
                X_test_w.append(window)
                y_test_w.append(label)

# Convert to NumPy arrays
X_train_w, y_train_w = np.array(X_train_w), np.array(y_train_w)
X_test_w, y_test_w = np.array(X_test_w), np.array(y_test_w)


# =====================================================================
# VERIFICATION REPORTS
# =====================================================================
def report_class_proportions(y_set, set_name):
    counts = pd.Series(y_set).value_counts()
    props = pd.Series(y_set).value_counts(normalize=True) * 100
    print(f"--- Class distribution in: {set_name} ---")
    for cls in ['N', 'S', 'V']:
        if cls in counts.index:
            print(f"  Class {cls}: {counts[cls]:>6} beats ({props[cls]:.2f}%)")
    print(f"  Total: {len(y_set)} beats\n")

print("==================================================")
print("CHECKING DIVISIONS (SINGLE BEAT)")
print("==================================================")
report_class_proportions(y_train_s, "Train Set (Single Beat)")
report_class_proportions(y_test_s, "Test Set (Single Beat)")

print("==================================================")
print("CHECKING DIVISIONS (SLIDING WINDOWS - CORRECTED)")
print("==================================================")
print(f"Shape of X_train_w: {X_train_w.shape} -> (Samples, T=10, Features=8)")
print(f"Shape of X_test_w:  {X_test_w.shape} -> (Samples, T=10, Features=8)\n")
report_class_proportions(y_train_w, "Train Set (Sliding Windows)")
report_class_proportions(y_test_w, "Test Set (Sliding Windows)")

CHECKING DIVISIONS (SINGLE BEAT)
--- Class distribution in: Train Set (Single Beat) ---
  Class N:  22729 beats (94.20%)
  Class S:    165 beats (0.68%)
  Class V:   1235 beats (5.12%)
  Total: 24129 beats

--- Class distribution in: Test Set (Single Beat) ---
  Class N:   5683 beats (94.20%)
  Class S:     41 beats (0.68%)
  Class V:    309 beats (5.12%)
  Total: 6033 beats

CHECKING DIVISIONS (SLIDING WINDOWS - CORRECTED)
Shape of X_train_w: (24027, 10, 8) -> (Samples, T=10, Features=8)
Shape of X_test_w:  (6009, 10, 8) -> (Samples, T=10, Features=8)

--- Class distribution in: Train Set (Sliding Windows) ---
  Class N:  22628 beats (94.18%)
  Class S:    165 beats (0.69%)
  Class V:   1234 beats (5.14%)
  Total: 24027 beats

--- Class distribution in: Test Set (Sliding Windows) ---
  Class N:   5661 beats (94.21%)
  Class S:     40 beats (0.67%)
  Class V:    308 beats (5.13%)
  Total: 6009 beats



In [17]:
# We store the three-dimensional arrays 
np.save('../data/3_processed/X_train_w.npy', X_train_w)
np.save('../data/3_processed/y_train_w.npy', y_train_w)
np.save('../data/3_processed/X_test_w.npy', X_test_w)
np.save('../data/3_processed/y_test_w.npy', y_test_w)

# We also save the individual arrays
'''
np.save('../data/3_processed/X_train_s.npy', X_train_s)
np.save('../data/3_processed/y_train_s.npy', y_train_s)
np.save('../data/3_processed/X_test_s.npy', X_test_s)
np.save('../data/3_processed/y_test_s.npy', y_test_s)
'''

"\nnp.save('../data/3_processed/X_train_s.npy', X_train_s)\nnp.save('../data/3_processed/y_train_s.npy', y_train_s)\nnp.save('../data/3_processed/X_test_s.npy', X_test_s)\nnp.save('../data/3_processed/y_test_s.npy', y_test_s)\n"